# Tetris ASCII
Execute a célula, clique em **Iniciar** e depois no desenho. Use as setas ou botões.

Python exibe HTML; JavaScript controla o jogo no navegador. Não precisa instalar pacotes.

A cobrinha atravessa as bordas e o próprio corpo. O Tetris usa blocos ASCII, o Pong é contra o computador e o RPG pede uma chave para abrir a porta.

In [ ]:
from IPython.display import HTML, display
from html import escape

# Python mostra a saída; JavaScript anima o ASCII e recebe as setas.
jogo = r"""<!doctype html><html lang="pt-BR"><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1"><title>Tetris ASCII</title>
<style>body{margin:0;background:#111;color:#a6ff99;padding:12px;font:14px monospace}pre{font:clamp(11px,3vw,17px)/1.2 monospace;white-space:pre;overflow:auto;outline:none;max-width:100%;margin:12px 0}pre:focus{outline:2px solid #a6ff99;outline-offset:3px}button{background:#223b22;color:#caffbc;border:1px solid #75956b;border-radius:5px;padding:9px;margin:3px;cursor:pointer;touch-action:manipulation}button:focus-visible{outline:2px solid white}#status{font-size:12px}h2{font-size:16px}</style>
<h2>Tetris ASCII</h2><button id="inicio">Iniciar / reiniciar</button><button id="pausa">Pausar / continuar</button><span id="status" role="status">Toque em Iniciar.</span>
<pre id="tela" tabindex="0" aria-label="Tetris ASCII. Clique aqui e use as setas."></pre>
<div aria-label="Controles de toque"><button data-key="ArrowUp" aria-label="Cima">Cima</button><button data-key="ArrowDown" aria-label="Baixo">Baixo</button><button data-key="ArrowLeft" aria-label="Esquerda">Esq.</button><button data-key="ArrowRight" aria-label="Direita">Dir.</button><button data-key=" " aria-label="Soltar peça">Soltar</button></div>
<script>
'use strict';
const tela=document.getElementById('tela'), status=document.getElementById('status');
let tempo=null, rodando=false, terminou=false;
function moldura(linhas){const borda='+'+'-'.repeat(linhas[0].length)+'+';return borda+'\n'+linhas.map(l=>'|'+l+'|').join('\n')+'\n'+borda;}
// O tabuleiro é uma matriz. # é uma peça fixa e @ é a peça caindo.
const largura = 10, altura = 16;
const formas = [ [[1,1,1,1]], [[1,1],[1,1]], [[0,1,0],[1,1,1]], [[1,0,0],[1,1,1]], [[0,0,1],[1,1,1]], [[0,1,1],[1,1,0]], [[1,1,0],[0,1,1]] ];
let tabuleiro, peca, x, y, pontos;
function cabe(px, py, forma) {
    return forma.every((linha, dy) => linha.every((bloco, dx) => !bloco ||
        (px+dx >= 0 && px+dx < largura && py+dy >= 0 && py+dy < altura && !tabuleiro[py+dy][px+dx])));
}
function novaPeca() {
    peca = formas[Math.floor(Math.random()*formas.length)].map(linha => [...linha]);
    x = Math.floor((largura - peca[0].length) / 2); y = 0;
    if (!cabe(x,y,peca)) terminou = true;
}
function reiniciar() {
    tabuleiro = Array.from({length:altura}, () => Array(largura).fill(0));
    pontos = 0; novaPeca();
}
function fixar() {
    peca.forEach((linha,dy) => linha.forEach((bloco,dx) => { if (bloco) tabuleiro[y+dy][x+dx] = 1; }));
    const restantes = tabuleiro.filter(linha => linha.some(bloco => bloco === 0));
    pontos += (altura - restantes.length) * 100;
    while (restantes.length < altura) restantes.unshift(Array(largura).fill(0));
    tabuleiro = restantes; novaPeca();
}
function atualizar() {
    if (cabe(x,y+1,peca)) y++;
    else fixar();
}
function comando(tecla) {
    if (tecla === 'ArrowLeft' && cabe(x-1,y,peca)) x--;
    if (tecla === 'ArrowRight' && cabe(x+1,y,peca)) x++;
    if (tecla === 'ArrowDown') atualizar();
    if (tecla === 'ArrowUp') {
        const girada = peca[0].map((_,coluna) => peca.map(linha => linha[coluna]).reverse());
        if (cabe(x,y,girada)) peca = girada;
    }
    if (tecla === ' ') {
        while (cabe(x,y+1,peca)) y++;
        fixar();
    }
}
function desenhar() {
    const desenho = tabuleiro.map(linha => linha.map(bloco => bloco ? '#' : '.'));
    if (!terminou) peca.forEach((linha,dy) => linha.forEach((bloco,dx) => { if (bloco) desenho[y+dy][x+dx] = '@'; }));
    return moldura(desenho.map(linha => linha.join(''))) + '\nPontos: ' + pontos + '\n' + (terminou ? 'FIM DE JOGO: o tabuleiro encheu.' : 'Cima: girar | Baixo: descer\nEsquerda/direita: mover | Espaco: soltar');
}

function mostrar(){tela.textContent=desenhar();if(terminou){parar();status.textContent='Partida encerrada. Toque em Iniciar para jogar outra vez.';}}
function parar(){clearInterval(tempo);tempo=null;rodando=false;}
function continuar(){if(terminou)return;parar();rodando=true;status.textContent='Jogando. Clique no desenho para usar as setas.';tempo=setInterval(()=>{atualizar();mostrar();},500);tela.focus();}
function iniciar(){parar();terminou=false;reiniciar();mostrar();continuar();}
function receber(tecla){if(!rodando||terminou)return;comando(tecla);mostrar();}
tela.addEventListener('keydown',e=>{if(['ArrowUp','ArrowDown','ArrowLeft','ArrowRight',' '].includes(e.key)){e.preventDefault();receber(e.key);}});
document.querySelectorAll('[data-key]').forEach(b=>b.onclick=()=>{receber(b.dataset.key);tela.focus();});
document.getElementById('inicio').onclick=iniciar;
document.getElementById('pausa').onclick=()=>{if(rodando){parar();status.textContent='Pausado.';}else continuar();};
window.addEventListener('pagehide',parar);
document.addEventListener('visibilitychange',()=>{if(document.hidden&&rodando){parar();status.textContent='Pausado. Toque em continuar.';}});
reiniciar();mostrar();
</script></html>"""

display(HTML('<iframe title="Tetris ASCII" sandbox="allow-scripts" style="width:100%;height:620px;border:0" srcdoc="' + escape(jogo, quote=True) + '"></iframe>'))